# Fine-tuning the Cd head

A follow-up to `train_model.ipynb`'s negative result: training a Cd-vs-pressure
consistency term jointly with everything else made both heads worse, because the
"teacher" signal (Cd derived from predicted pressure) was unreliable early on and
its noise leaked into the shared encoder through `cd_pred`'s path.

The fix here: start from the already fully-trained model, **freeze the encoder and
pressure head completely**, and fine-tune only the small Cd-head MLP. Two things
change versus the joint-training attempt: the pressure predictions used as the
consistency teacher are now accurate from the very first step (no warmup needed),
and there is no shared-encoder path left for a bad gradient to leak through, since
the encoder's parameters are frozen -- whatever happens to the Cd head here cannot
touch pressure accuracy.

In [1]:
import sys, json, copy
sys.path.insert(0, "..")

import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader

from src.model import MeshSurrogate
from src.shape_opt import differentiable_drag_proxy_batch

device = "cuda" if torch.cuda.is_available() else "cpu"

with open("../outputs/norm_stats.json") as f:
    stats = json.load(f)
pressure_mean, pressure_std = stats["pressure_mean"], stats["pressure_std"]
cd_mean, cd_std = stats["cd_mean"], stats["cd_std"]

train_data = torch.load("../outputs/cache/train_pyg.pt", weights_only=False)
val_data = torch.load("../outputs/cache/val_pyg.pt", weights_only=False)
test_data = torch.load("../outputs/cache/test_pyg.pt", weights_only=False)
train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
val_loader = DataLoader(val_data, batch_size=4)
test_loader = DataLoader(test_data, batch_size=4)

model = MeshSurrogate(in_channels=6).to(device)
model.load_state_dict(torch.load("../outputs/checkpoints/best_model.pt", weights_only=True, map_location=device))
print("loaded base checkpoint")

loaded base checkpoint


Freeze everything except the Cd head. `model.encode(...)` and the pressure head keep
running forward normally (their outputs are still needed as inputs to the Cd head
and as the consistency teacher) -- they just don't accumulate gradients or get
updated by this optimizer.

In [2]:
for p in model.convs.parameters():
    p.requires_grad_(False)
for p in model.bns.parameters():
    p.requires_grad_(False)
for p in model.pressure_head.parameters():
    p.requires_grad_(False)

trainable = list(model.cd_head.parameters())
print("trainable params:", sum(p.numel() for p in trainable), "(cd head only)")

optimizer = torch.optim.Adam(trainable, lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)
BETA_CONSISTENCY = 1.0

trainable params: 65793 (cd head only)


One fine-tuning epoch: `MSE(cd_pred, true_cd)` keeps the head from drifting away
from the real (if sparse) labels, `smooth_l1(cd_pred, analytic_cd(pressure_pred))`
pulls it toward agreement with the frozen, accurate pressure head.

In [3]:
def run_epoch(loader, train: bool):
    model.train(train)
    total_cd_loss, total_cons_loss, n = 0.0, 0.0, 0
    for batch in loader:
        batch = batch.to(device)
        if train:
            optimizer.zero_grad()
        pressure_pred, cd_pred = model(batch.x, batch.edge_index, batch.batch)
        cd_loss = F.mse_loss(cd_pred, batch.y_cd.squeeze(-1))

        pressure_pred_raw = pressure_pred.detach() * pressure_std + pressure_mean
        analytic_cd_raw = differentiable_drag_proxy_batch(
            batch.x[:, :3], batch.batch, batch.face, batch.face_batch, pressure_pred_raw
        )
        analytic_cd_norm = (analytic_cd_raw - cd_mean) / cd_std
        cons_loss = F.smooth_l1_loss(cd_pred, analytic_cd_norm)

        loss = cd_loss + BETA_CONSISTENCY * cons_loss
        if train:
            loss.backward()
            optimizer.step()
        bs = batch.num_graphs
        total_cd_loss += cd_loss.item() * bs
        total_cons_loss += cons_loss.item() * bs
        n += bs
    return (total_cd_loss / n) * (cd_std ** 2), (total_cons_loss / n) * (cd_std ** 2)

Fine-tune for up to 100 epochs, keeping the state dict with the best validation Cd
MSE (not the best combined loss -- we care about not regressing the real label fit,
the consistency term is a regularizer, not the target metric).

In [4]:
N_EPOCHS = 100
best_val_cd = float("inf")
best_state = None
history = {"train_cd": [], "val_cd": [], "train_cons": [], "val_cons": []}

for epoch in range(N_EPOCHS):
    train_cd, train_cons = run_epoch(train_loader, train=True)
    with torch.no_grad():
        val_cd, val_cons = run_epoch(val_loader, train=False)
    scheduler.step(val_cd)

    history["train_cd"].append(train_cd)
    history["val_cd"].append(val_cd)
    history["train_cons"].append(train_cons)
    history["val_cons"].append(val_cons)

    if val_cd < best_val_cd:
        best_val_cd = val_cd
        best_state = copy.deepcopy(model.state_dict())

    if epoch % 10 == 0 or epoch == N_EPOCHS - 1:
        print(f"epoch {epoch:3d}  train_cd {train_cd:.4f}  val_cd {val_cd:.4f}  train_cons {train_cons:.4f}  val_cons {val_cons:.4f}")

print("best val_cd:", best_val_cd)

epoch   0  train_cd 1.5290  val_cd 4.1546  train_cons 5.0731  val_cons 7.5628


epoch  10  train_cd 0.8348  val_cd 6.1295  train_cons 3.6073  val_cons 6.4228


epoch  20  train_cd 0.9138  val_cd 5.7160  train_cons 3.3629  val_cons 6.0466


epoch  30  train_cd 0.7790  val_cd 4.8733  train_cons 3.5292  val_cons 6.7349


epoch  40  train_cd 0.6982  val_cd 5.1031  train_cons 3.1772  val_cons 6.3654


...

epoch  60  train_cd 0.6615  val_cd 5.0053  train_cons 3.4593  val_cons 6.3666


epoch  70  train_cd 0.6530  val_cd 5.1791  train_cons 3.0740  val_cons 6.0611


epoch  80  train_cd 0.6230  val_cd 5.2440  train_cons 3.1095  val_cons 6.3003


epoch  90  train_cd 0.6757  val_cd 4.9235  train_cons 3.0937  val_cons 6.3578


epoch  99  train_cd 0.7184  val_cd 5.2747  train_cons 3.3552  val_cons 5.8525
best val_cd: 4.154568946654056


Compare against the pre-fine-tune baseline on the test set, and only keep the
fine-tuned checkpoint if it's actually at least as good on real Cd accuracy --
otherwise this whole exercise should be judged a second negative result and the
original checkpoint left alone.

In [5]:
baseline_state = torch.load("../outputs/checkpoints/best_model.pt", weights_only=True, map_location=device)
model.load_state_dict(baseline_state)
with torch.no_grad():
    test_cd_before, test_cons_before = run_epoch(test_loader, train=False)

model.load_state_dict(best_state)
with torch.no_grad():
    test_cd_after, test_cons_after = run_epoch(test_loader, train=False)

print(f"test Cd MSE:            before {test_cd_before:.4f}  ->  after {test_cd_after:.4f}")
print(f"test consistency MSE:   before {test_cons_before:.4f}  ->  after {test_cons_after:.4f}")

if test_cd_after <= test_cd_before * 1.05:
    torch.save(best_state, "../outputs/checkpoints/best_model.pt")
    print("fine-tuned Cd head kept (real Cd accuracy did not regress) -- checkpoint updated")
else:
    print("fine-tuned Cd head REJECTED (real Cd accuracy regressed) -- original checkpoint left untouched")

test Cd MSE:            before 4.4800  ->  after 4.8914
test consistency MSE:   before 6.8876  ->  after 6.6538
fine-tuned Cd head REJECTED (real Cd accuracy regressed) -- original checkpoint left untouched
